# Assignment 1 Perceptron Image Classification
### Arseniy Uspenskiy
### USPARS001

Import necessary packages

In [26]:
import numpy as np
from PIL import Image
import os
from sklearn.model_selection import train_test_split

## Data Processing

Creating method for loading all images of a specified directory

Images are flattened into vectors and returned alongside their character label

In [27]:
CHARACTERS = ["bart_simpson", "charles_montgomery_burns", "homer_simpson",
    "krusty_the_clown", "lisa_simpson", "marge_simpson",
    "milhouse_van_houten", "moe_szyslak", "ned_flanders", "principal_skinner"]

# Method for loading images from a particular directory e.g.(rgb-train)
def load_dataset(path):

    vectors = []
    labels = []

    for name in CHARACTERS:
        char_dir = os.path.join(path, name)

        for file in os.listdir(char_dir):

            img_path = os.path.join(char_dir, file)
            img = Image.open(img_path)
            arr = np.array(img)
            vectors.append(arr.flatten())
            labels.append(CHARACTERS.index(name))

    x = np.array(vectors)
    y = np.array(labels)

    return x, y

Loading seperate training, validation, and test datasets for both rgb and grayscale

In [28]:
X_train_gray_full, y_train_gray_full = load_dataset("grayscale-train")
X_test_gray, y_test_gray = load_dataset("grayscale-test")

X_train_rgb_full, y_train_rgb_full = load_dataset("rgb-train")
X_test_rgb, y_test_rgb = load_dataset("rgb-test")

X_train_gray, X_val_gray, y_train_gray, y_val_gray = train_test_split(X_train_gray_full, y_train_gray_full, test_size=0.2, stratify=y_train_gray_full, random_state=42)
X_train_rgb, X_val_rgb, y_train_rgb, y_val_rgb = train_test_split(X_train_rgb_full, y_train_rgb_full, test_size=0.2, stratify=y_train_rgb_full, random_state=42)

## Multi-class perceptron implementation

BinaryPerceptron with a predict method for outputting 0 or 1

In [48]:

class BinaryPerceptron:

    def __init__(self, id, n, alpha=0.05):
        self.id = id
        self.weights = np.ones(n, dtype=float) / 10
        self.bias = 0.1
        self.alpha = alpha

    def score(self, x):
            weighted_sum = np.dot(x, self.weights) + self.bias
            return weighted_sum

    def predict(self, x):
        return 1 if self.score(x) >= 0 else 0

    def learning_rule(self, x, y):
        g = self.predict(x)
        self.weights = self.weights + self.alpha * (y - g) * (x / 255)
        self.bias = self.bias + self.alpha * (y - g)

    def __repr__(self):
        return CHARACTERS[self.id] + " Perceptron"

MultiClassPerceptron which utilizes a one versus rest approach using 10 binary perceptrons to classify images

In [46]:
class MultiClassPerceptron:

    def __init__(self, n):
        self.perceptrons = [BinaryPerceptron(i, n) for i in range(10)]

    def predict(self, x):
        top_score = float("-inf")
        top_num = 0
        for i in range(len(self.perceptrons)):
            y = self.perceptrons[i].score(x)
            if y > top_score:
                top_score = y
                top_num = i

        return top_num

    def train_perceptron(self, X, y):
        for i in range(len(X)):
            for percep in self.perceptrons:
                y_converted = 0
                if percep.id == y[i]:
                    y_converted = 1
                percep.learning_rule(X[i], y_converted)


## Training

Method for assesing accuracy of a perceptron

In [31]:
def accuracy(perceptron: BinaryPerceptron, data_X, data_y):
    correct = 0
    total = 0

    for X, y in zip(data_X, data_y):
        y_hat = perceptron.predict(X)
        if y_hat == y:
            correct += 1
        total += 1

    accuracy = correct / total
    return accuracy

Training Perceptron over multiple epochs

Exit Conditions: max epochs reached or insignificant improvements in accuracy

In [ ]:
p_gray = MultiClassPerceptron(len(X_train_gray[0]))
p_rgb = MultiClassPerceptron(len(X_train_rgb[0]))
epochs = 25

print("Starting Training")
print()

for epoch in range(epochs):

    perm = np.random.permutation(len(X_train_gray))
    X_train_gray_shuff = X_train_gray[perm]
    y_train_gray_shuff = y_train_gray[perm]
    X_train_rgb_shuff = X_train_rgb[perm]
    y_train_rgb_shuff = y_train_rgb[perm]


    p_gray.train_perceptron(X_train_gray_shuff, y_train_gray_shuff)
    p_rgb.train_perceptron(X_train_rgb_shuff, y_train_rgb_shuff)

    acc_gray = accuracy(p_gray, X_val_gray, y_val_gray)
    acc_rgb = accuracy(p_rgb, X_val_rgb, y_val_rgb)

    print(f"Epoch {epoch+1}: validation accuracy for gray = {acc_gray:.4f}")
    print(f"Epoch {epoch+1}: validation accuracy for rgb = {acc_rgb:.4f}")
    print()